<a href="https://colab.research.google.com/github/jyryu3161/bioinfo/blob/main/01_tcga_brca_pam50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCSC Xena 데이터 전처리


In [3]:
import pandas as pd
import numpy as np

# 파일 경로 설정
expr_file_path = '/content/TCGA-BRCA.star_tpm.tsv.gz'
pam50_file_path = '/content/brca_pam50'
output_file_path = '/content/TCGA-BRCA_filtered_subset.csv'

# 1. 발현 데이터 로드
try:
    expr_df = pd.read_csv(
        expr_file_path,
        sep='\t',
        index_col=0,
        compression='gzip',
        na_values=['NA', 'NaN', '']
    )
    print(f"'{expr_file_path}' 파일을 로드했습니다. 데이터 크기: {expr_df.shape}")

except Exception as e:
    print(f"발현 데이터 로드 오류: {e}")
    expr_df = None

# 2. brca_pam50 파일에서 환자 ID 추출
try:
    pam50_patients = set()

    with open(pam50_file_path, 'r') as f:
        for line in f:
            line = line.strip()

            if line:
                patient_id = line.split('\t')[0][:12]
                pam50_patients.add(patient_id)

    print(f"'{pam50_file_path}'에서 고유 환자 ID {len(pam50_patients)}개를 추출했습니다.")

except Exception as e:
    print(f"PAM50 파일 로드 오류: {e}")
    pam50_patients = set()

# 3. 필터링 및 전처리 수행
if expr_df is not None and pam50_patients:

    # Ensembl_ID 버전 제거
    expr_df.index = expr_df.index.str.split('.').str[0]

    # 중복 row 제거
    expr_df = expr_df[~expr_df.index.duplicated(keep='first')]

    print(f"중복 제거 후 데이터 크기: {expr_df.shape}")

    # 발현 데이터 컬럼에서 환자 ID(12자리) 추출
    filtered_samples = [
        col for col in expr_df.columns
        if col[:12] in pam50_patients
    ]

    if filtered_samples:

        expr_df_subset = expr_df[filtered_samples]

        print(f"\n필터링 완료! 새로운 데이터 크기: {expr_df_subset.shape}")

        # 저장
        expr_df_subset.to_csv(output_file_path)

        print(f"결과 파일이 '{output_file_path}'에 저장되었습니다.")

        display(expr_df_subset.head())

    else:
        print("\n일치하는 샘플을 찾을 수 없습니다.")
        expr_df_subset = pd.DataFrame()

else:
    print("\n데이터가 로드되지 않았거나 환자 목록이 비어있습니다.")

'/content/TCGA-BRCA.star_tpm.tsv.gz' 파일을 로드했습니다. 데이터 크기: (60660, 1226)
'/content/brca_pam50'에서 고유 환자 ID 75개를 추출했습니다.
중복 제거 후 데이터 크기: (60616, 1226)

필터링 완료! 새로운 데이터 크기: (60616, 87)
결과 파일이 '/content/TCGA-BRCA_filtered_subset.csv'에 저장되었습니다.


,TCGA-AO-A0J5-01A,TCGA-BH-A0B1-01A,TCGA-AO-A03N-01B,TCGA-3C-AALJ-01A,TCGA-A2-A0YT-01A,TCGA-BH-A0B5-01A,TCGA-C8-A12Q-01A,TCGA-A2-A0EV-01A,TCGA-AR-A0U4-01A,TCGA-A7-A0CH-11A,...,TCGA-C8-A130-01A,TCGA-BH-A0DL-11A,TCGA-AQ-A04L-01B,TCGA-A2-A0T7-01A,TCGA-D8-A13Z-01A,TCGA-AO-A12F-01A,TCGA-A2-A0YK-01A,TCGA-BH-A0DV-11A,TCGA-A2-A0T5-01A,TCGA-AR-A0U0-01A
Ensembl_ID,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,4.645690,6.021755,3.613025,5.348923,4.223245,5.451732,5.143761,5.039169,5.570548,5.818705,...,5.580664,5.930872,5.349026,5.159674,7.011063,6.555207,5.369414,6.581098,6.116556,4.572532
ENSG00000000005,1.255622,0.392647,0.000000,2.554785,0.096532,0.619554,1.293017,1.589140,2.026517,6.187453,...,0.912727,5.311957,0.196481,0.539531,0.099497,2.075019,1.726134,3.192368,0.059632,0.000000
ENSG00000000419,5.749910,6.977449,6.079559,7.208299,7.462536,6.701132,7.245631,6.800720,7.254893,6.080071,...,6.871424,6.349917,7.623860,7.981983,7.269562,7.269970,6.518782,7.085236,6.464366,6.816136
ENSG00000000457,4.487107,4.021000,2.287502,3.111148,4.515176,4.957734,4.338146,4.022536,3.807685,2.906256,...,2.981232,3.754481,3.800796,4.510481,4.501605,4.067622,4.485124,4.282692,4.353316,3.232492
ENSG00000000460,2.545178,3.372492,1.753177,2.709401,2.938982,4.548548,3.223716,2.927536,3.729009,1.519139,...,2.854375,2.200755,2.026765,2.985664,4.894866,3.452055,2.916973,2.601411,3.460140,2.915769
